In [ ]:
"""
Microsoft Agent Framework (MAF) - Reasoning Model Parameters Example

This demonstrates how to use the reasoning_effort parameter with the
Microsoft Agent Framework against Groq's OpenAI-compatible Chat Completions
endpoint, contrasting two effort levels:
1. medium reasoning effort
2. high reasoning effort
"""
import asyncio
import os

from dotenv import load_dotenv


# ============================================================================
# TEST 1: Chat Completions API (medium reasoning effort)
# ============================================================================
# Uses OpenAIChatCompletionClient (Groq) with additional_chat_options for
# reasoning params. Endpoint: POST /chat/completions


async def test_chat_completions_api():
    """
    Test Chat Completions API with reasoning parameters (medium effort).

    Uses additional_chat_options to pass Groq's flat reasoning parameter:
    - reasoning_effort: low, medium, high
    """
    from agent_framework import Agent
    from agent_framework.openai import OpenAIChatCompletionClient

    print("\n" + "-" * 50)
    print("TEST 1: Chat Completions API (reasoning_effort=medium)")
    print("-" * 50)
    api_key = os.getenv("GROQ_API_KEY")
    model = "openai/gpt-oss-20b"

    # Instantiate Groq client via OpenAI Chat Completions-compatible endpoint
    client = OpenAIChatCompletionClient(
        model=model,
        api_key=api_key,
        base_url="https://api.groq.com/openai/v1",
    )

    # Create agent with reasoning parameters via default_options
    agent = Agent(
        client=client,
        name="chat-completions-agent",
        instructions="You are a helpful reasoning assistant. Think step by step.",
        default_options={
            "max_tokens": 4096,
            "reasoning_effort": "medium",
        },
    )

    question = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?"

    print(f"Question: {question}\n")
    response = await agent.run(question)
    print(f"Response: {response.text}")

    return response


# ============================================================================
# TEST 2: Chat Completions API (high reasoning effort)
# ============================================================================
# Same Groq Chat Completions endpoint as Test 1, but with reasoning_effort=high
# to contrast how effort level affects the model's reasoning.


async def test_high_effort_api():
    """
    Test Chat Completions API with reasoning_effort=high.

    Mirrors Test 1 but raises the reasoning effort to observe the difference
    in the model's step-by-step reasoning.
    """
    from agent_framework import Agent
    from agent_framework.openai import OpenAIChatCompletionClient

    print("\n" + "-" * 50)
    print("TEST 2: Chat Completions API (reasoning_effort=high)")
    print("-" * 50)
    api_key = os.getenv("GROQ_API_KEY")
    model = "openai/gpt-oss-20b"

    # Instantiate Groq client via OpenAI Chat Completions-compatible endpoint
    client = OpenAIChatCompletionClient(
        model=model,
        api_key=api_key,
        base_url="https://api.groq.com/openai/v1",
    )

    # Create agent with high reasoning effort
    agent = Agent(
        client=client,
        name="high-effort-agent",
        instructions="You are an expert problem solver. Think carefully before answering.",
        default_options={
            "max_tokens": 4096,
            "reasoning_effort": "high",
        },
    )

    question = "What is 25 * 17? Show your reasoning step by step."

    print(f"Question: {question}\n")
    response = await agent.run(question)
    print(f"Response: {response.text}")

    return response


# ============================================================================
# Main execution
# ============================================================================


async def main():
    load_dotenv()

    print("=" * 60)
    print("Microsoft Agent Framework - Reasoning Parameters Demo")
    print("=" * 60)

    # Test 1: Chat Completions API (medium effort)
     await test_chat_completions_api()

    print("\n")

    # Test 2: Chat Completions API (high effort)
    # await test_high_effort_api()

    print("\n" + "=" * 60)
    print("Both tests completed!")
    print("=" * 60)

In [ ]:
#!/usr/bin/env python3
"""
launch_all.py — one-command launcher for the agent-mesh stack.

Opens each service in its OWN terminal window so you can watch its logs live,
and starts them in the right order (MCP servers first, then the mesh that
connects to them, then the REST bridge, then the React UI).

Usage:
    python launch_all.py

Place this file in the ROOT folder that contains:
    datalayer-as-service/
    rag-as-a-service/
    agent-mesh/

Stop everything by closing the individual windows (Ctrl+C in each), or run:
    python launch_all.py --stop        (best-effort: closes windows by title)
"""

import os
import sys
import time
import shutil
import platform
import subprocess
from pathlib import Path

# --------------------------------------------------------------------------
# Config: each service = (window title, working dir, command, env overrides, wait_after)
# `wait_after` = seconds to pause AFTER launching, before starting the next one.
# --------------------------------------------------------------------------
ROOT = Path(__file__).resolve().parent
IS_WINDOWS = platform.system() == "Windows"

# Use the same Python interpreter that's running this script.
PY = sys.executable or "python"

SERVICES = [
    {
        "title": "DataLayer-MCP",
        "cwd": ROOT / "datalayer-as-service",
        "cmd": [PY, "-m", "mcp_server.server"],
        "env": {"MCP_TRANSPORT": "http", "MCP_HOST": "127.0.0.1", "MCP_PORT": "9100"},
        "wait_after": 4,   # give the MCP server time to bind its port
    },
    {
        "title": "RAG-MCP",
        "cwd": ROOT / "rag-as-a-service",
        "cmd": [PY, "-m", "mcp_integration.server"],
        "env": {"MCP_TRANSPORT": "http", "MCP_HOST": "127.0.0.1", "MCP_PORT": "9000"},
        "wait_after": 4,
    },
    {
        "title": "Agent-Mesh",
        "cwd": ROOT / "agent-mesh",
        "cmd": [PY, "launch_mesh.py"],
        "env": {},
        "wait_after": 5,   # let the 4 A2A nodes register before the REST bridge
    },
    {
        "title": "REST-API",
        "cwd": ROOT / "agent-mesh",
        "cmd": [PY, "api_server.py"],
        "env": {},
        "wait_after": 3,
    },
    {
        "title": "React-UI",
        "cwd": ROOT / "agent-mesh" / "frontend",
        "cmd": ["npm", "run", "dev"],
        "env": {},
        "wait_after": 0,
    },
]


def _preflight():
    """Verify folders and key files exist before launching anything."""
    problems = []
    for svc in SERVICES:
        if not svc["cwd"].is_dir():
            problems.append(f"  - Missing folder: {svc['cwd']}")
    if not IS_WINDOWS and shutil.which("npm") is None:
        problems.append("  - 'npm' not found on PATH (needed for React UI)")
    if problems:
        print("Pre-flight check failed:\n" + "\n".join(problems))
        print("\nRun this script from the folder that contains the service directories.")
        sys.exit(1)


def _launch_windows(svc):
    """Open a new PowerShell window, cd into cwd, set env vars, run the command."""
    env_prefix = "".join(f'$env:{k}="{v}"; ' for k, v in svc["env"].items())
    cmd_str = subprocess.list2cmdline(svc["cmd"])
    inner = f'{env_prefix}{cmd_str}'
    # -NoExit keeps the window open so you can read logs / errors.
    ps_args = [
        "powershell", "-NoExit", "-Command",
        f'$host.UI.RawUI.WindowTitle = "{svc["title"]}"; '
        f'Set-Location -Path "{svc["cwd"]}"; {inner}',
    ]
    subprocess.Popen(["cmd", "/c", "start", svc["title"], *ps_args])


def _launch_unix(svc):
    """Open a new terminal on macOS/Linux."""
    env_prefix = " ".join(f'{k}="{v}"' for k, v in svc["env"].items())
    cmd_str = " ".join(subprocess.list2cmdline([c]) for c in svc["cmd"])
    full = f'cd "{svc["cwd"]}" && {env_prefix} {cmd_str}'.strip()

    if platform.system() == "Darwin":
        # macOS Terminal via AppleScript
        script = f'tell application "Terminal" to do script "{full}"'
        subprocess.Popen(["osascript", "-e", script])
    else:
        # Linux: try a few common terminals
        for term in (["gnome-terminal", "--", "bash", "-c", f"{full}; exec bash"],
                     ["konsole", "-e", "bash", "-c", f"{full}; exec bash"],
                     ["xterm", "-e", f"{full}; exec bash"]):
            if shutil.which(term[0]):
                subprocess.Popen(term)
                return
        print(f"  ! No supported terminal found; run manually:\n    {full}")


def main():
    if "--stop" in sys.argv and IS_WINDOWS:
        for svc in SERVICES:
            subprocess.run(["taskkill", "/FI", f'WINDOWTITLE eq {svc["title"]}*', "/F"],
                           capture_output=True)
        print("Sent stop signal to service windows.")
        return

    _preflight()
    print(f"Launching {len(SERVICES)} services from: {ROOT}\n")

    for svc in SERVICES:
        print(f"  -> {svc['title']:14} ({svc['cwd'].name})")
        if IS_WINDOWS:
            _launch_windows(svc)
        else:
            _launch_unix(svc)
        if svc["wait_after"]:
            time.sleep(svc["wait_after"])

    print("\nAll services launched in separate windows.")
    print("Watch each window for logs. Close a window (or Ctrl+C in it) to stop that service.")
    if IS_WINDOWS:
        print("To stop all at once:  python launch_all.py --stop")


if __name__ == "__main__":
    main()

In [ ]:
# In a Jupyter kernel an event loop is already running, so use await instead of asyncio.run()
await main()

Microsoft Agent Framework - Reasoning Parameters Demo



--------------------------------------------------
TEST 2: Chat Completions API (reasoning_effort=high)
--------------------------------------------------
Question: What is 25 * 17? Show your reasoning step by step.

Response: To multiply \(25\) by \(17\), you can use the distributive property (breaking the number into parts) or the standard long‑multiplication method.  
Below are a couple of step‑by‑step ways to get the product.

---

## 1. Using the Distributive Property

\(17\) can be split into \(10 + 7\):

\[
\begin{aligned}
25 \times 17 &= 25 \times (10 + 7)\\
&= 25 \times 10 \;+\; 25 \times 7\\
&= 250 \;+\; 175\\
&= 425
\end{aligned}
\]

So \(25 \times 17 = 425\).

---

## 2. Using a Different Break‑Down

You can also split \(25\) into \(20 + 5\):

\[
\begin{aligned}
25 \times 17 &= (20 + 5) \times 17\\
&= 20 \times 17 \;+\; 5 \times 17\\
&= 340 \;+\; 85\\
&= 425
\end{aligned}
\]

Either way, the result is the same.

---



: 